# Jurisight – Part 3: Explainable AI (XAI)

This notebook demonstrates explainability for verdict prediction using SHAP or Integrated Gradients.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Install dependencies


In [ ]:
!pip -q install transformers torch captum shap

## Load the fine-tuned model


In [ ]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_DIR = Path('/content/drive/MyDrive/Jurisight/models/legal-bert')
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

## Integrated Gradients with Captum

This example highlights which tokens were most influential for the prediction.


In [ ]:
from captum.attr import IntegratedGradients

def predict_logits(input_ids, attention_mask):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    return outputs.logits

ig = IntegratedGradients(predict_logits)

sample_text = 'The applicant alleges a violation of Article 6 due to unfair trial procedures.'
inputs = tokenizer(sample_text, return_tensors='pt', truncation=True, max_length=512)
attributions, delta = ig.attribute(
    inputs=inputs['input_ids'],
    additional_forward_args=inputs['attention_mask'],
    target=None,
    return_convergence_delta=True
)
token_ids = inputs['input_ids'][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(token_ids)
token_attributions = attributions.sum(dim=-1).squeeze(0)
list(zip(tokens[:20], token_attributions[:20].detach().cpu().numpy()))

## SHAP (optional)

Use SHAP's `Explainer` for a more visual explanation. This can be expensive on long texts, so start with short inputs.


In [ ]:
import shap

def model_predict(texts):
    inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
    return probs

explainer = shap.Explainer(model_predict, tokenizer)
shap_values = explainer([sample_text])
shap.plots.text(shap_values[0])